# Module 04 — Learning to classify, and evaluating without fooling yourself

> **Aquatic Guardian · Learn computer vision** — Module 04 of 09.  
> **Time:** about 2.5 hours. **Runs on:** free Google Colab, CPU is enough. **Skip if:** you know train/validation/test splits, data leakage, precision/recall/F1, and confidence thresholds.

**How to use this notebook.** In Colab choose **File → Save a copy in Drive** first, so your work is kept.
Then **Runtime → Run all**. Every cell runs without errors before you touch anything; the exercises print
feedback rather than crashing. Cells are marked:

- ▶ **Run and read** — nothing to change.
- ✏️ **Predict, then change** — write down what you expect, change one value, re-run.
- 🧩 **Your turn** — replace `None` with your code. A checker tells you whether it works. Hidden solutions follow.

### By the end you can
1. Turn images into features and train a softmax (multinomial logistic) classifier from scratch.
2. Split video data **by clip/session**, and show what happens to your score if you don't.
3. Read a confusion matrix and compute precision, recall and F1 per class.
4. Explain why accuracy misleads on imbalanced data.
5. Add an **abstention** rule that returns `unknown` below a confidence threshold, and read the coverage trade-off.

This module is the backbone of **Mentee 1, Tasks 2–5, 7 and 9**.

In [ ]:
#@title ▶ Setup: scenes and real wildlife photographs (run it; you don't need to read it yet)
import numpy as np, matplotlib.pyplot as plt, urllib.request
from pathlib import Path
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10})

PHOTO_BASE = "https://raw.githubusercontent.com/ejri/aquatic-guardian-learning-modules/main/images/"
PHOTO_FILES = {
    "reef-school": "reef-school.jpg", "manatees": "manatees.jpg", "sea-lion": "sea-lion.jpg",
    "nurse-shark": "nurse-shark.jpg", "eagle-ray": "eagle-ray.jpg", "grouper": "grouper.jpg",
    "angelfish": "angelfish.jpg", "turtle": "turtle.jpg", "noaa-honu": "noaa-honu.jpg",
    "noaa-whale": "noaa-whale.jpg", "kelp-forest": "kelp-forest.jpg", "zebra-shark": "zebra-shark.jpg",
}

def load_photo(name, max_side=480):
    """Load a real wildlife JPEG. Looks in ./images and ../images, then downloads from GitHub."""
    file = PHOTO_FILES.get(name, name if str(name).endswith(".jpg") else str(name) + ".jpg")
    arr = None
    for candidate in (Path("images") / file, Path("../images") / file, Path(file)):
        if candidate.is_file():
            arr = plt.imread(candidate)
            break
    if arr is None:
        urllib.request.urlretrieve(PHOTO_BASE + file, file)
        arr = plt.imread(file)
    if getattr(arr, "dtype", None) == np.uint8:
        arr = arr.astype(float) / 255.0
    arr = np.asarray(arr[..., :3], dtype=float)
    h, w = arr.shape[:2]
    scale = max_side / max(h, w)
    if scale < 1:
        yy = (np.linspace(0, h - 1, max(1, int(h * scale)))).astype(int)
        xx = (np.linspace(0, w - 1, max(1, int(w * scale)))).astype(int)
        arr = arr[yy][:, xx]
    return arr


# Illustrative attenuation coefficients beta (1/m) for R, G, B, and the colour of the veiling light.
# These are round teaching numbers with the right ordering, not measurements of any real site.
WATER = {
    "clear_ocean":    dict(beta=(0.40, 0.07, 0.04), veil=(0.02, 0.20, 0.42)),
    "coastal":        dict(beta=(0.55, 0.20, 0.30), veil=(0.05, 0.32, 0.27)),
    "turbid_harbour": dict(beta=(0.95, 0.60, 0.75), veil=(0.14, 0.30, 0.22)),
}

def _fish_mask(H, W, cx, cy, L, ang, aspect=4.0):
    """Ellipse body plus a triangular tail. L is total length (snout to tail tip) in pixels."""
    yy, xx = np.mgrid[0:H, 0:W]
    a = L / 2.25; b = a / aspect
    X = (xx - cx) * np.cos(ang) + (yy - cy) * np.sin(ang)
    Y = -(xx - cx) * np.sin(ang) + (yy - cy) * np.cos(ang)
    body = (X / a) ** 2 + (Y / b) ** 2 <= 1
    tx = -X - 0.80 * a
    tail = (tx >= 0) & (tx <= 0.45 * a) & (np.abs(Y) <= 1.3 * b * tx / (0.45 * a))
    return body | tail, X, Y, a, b

FISH_COLOURS = [(0.97, 0.55, 0.12), (0.96, 0.84, 0.22), (0.80, 0.82, 0.84), (0.93, 0.35, 0.25)]

def make_scene(n_fish=4, H=120, W=180, seed=0, bg_seed=None, fish_dist=(2.0, 6.0), bg_dist=12.0,
               length_px=(22, 40), min_gap=0.55, touching=False, aspect=(4.0, 4.0)):
    """Returns a dict with the true scene radiance J (what you'd see in air), a distance map in metres,
    one boolean mask and one box (x1, y1, x2, y2) per fish, and per-fish truth."""
    r = np.random.default_rng(seed)
    rb = np.random.default_rng(seed if bg_seed is None else bg_seed)
    yy, xx = np.mgrid[0:H, 0:W] / max(H, W)
    J = np.stack([0.48 + 0.10 * yy, 0.43 + 0.08 * yy, 0.34 + 0.04 * xx], -1)       # reef wall / sand
    J = J + 0.04 * rb.standard_normal((H, W, 1)) + 0.015 * rb.standard_normal((H, W, 3))
    dist = np.full((H, W), float(bg_dist))
    masks, boxes, truth, placed = [], [], [], []
    for i in range(n_fish):
        for _ in range(200):
            L = r.uniform(*length_px); ang = r.uniform(-0.6, 0.6) + (np.pi if r.random() < 0.5 else 0)
            cx = r.uniform(0.15 * W, 0.85 * W); cy = r.uniform(0.18 * H, 0.82 * H)
            gap = 0.25 if touching else min_gap
            if all((cx - px) ** 2 + (cy - py) ** 2 > (gap * (L + pl)) ** 2 for px, py, pl in placed):
                break
        placed.append((cx, cy, L))
        asp = aspect[0] if aspect[0] == aspect[1] else r.uniform(*aspect)
        m, X, Y, a, b = _fish_mask(H, W, cx, cy, L, ang, asp)
        col = np.array(FISH_COLOURS[i % len(FISH_COLOURS)])
        J[m] = col
        stripe = m & (np.abs(X - 0.15 * a) < 0.09 * a)
        J[stripe] = (0.95, 0.95, 0.92)
        eye = (X - 0.62 * a) ** 2 + (Y - 0.15 * b) ** 2 < (0.12 * b + 0.6) ** 2
        J[eye & m] = (0.05, 0.05, 0.08)
        d = r.uniform(*fish_dist); dist[m] = d
        ys, xs = np.where(m)
        masks.append(m); boxes.append((int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1))
        truth.append(dict(cx=float(cx), cy=float(cy), length_px=float(L), angle=float(ang), dist_m=float(d), aspect=float(asp)))
    return dict(J=np.clip(J, 0, 1), dist=dist, masks=masks, boxes=boxes, truth=truth)

def underwater(J, dist, water="coastal", light=1.0, photons=None, seed=0):
    """Simple image-formation model, per colour channel c:
         I_c = light * ( J_c * t_c  +  B_c * (1 - t_c) ),   t_c = exp(-beta_c * distance)
    photons: if set, adds shot noise (fewer photons -> noisier). Lower `light` means fewer photons."""
    beta = np.array(WATER[water]["beta"]); B = np.array(WATER[water]["veil"])
    t = np.exp(-beta[None, None, :] * dist[..., None])
    I = light * (J * t + B * (1 - t))
    if photons:
        rng = np.random.default_rng(seed)
        n = photons * light
        I = rng.poisson(np.clip(I, 0, None) / max(light, 1e-6) * n) / n * light
    return np.clip(I, 0, 1)

def degrade(img, turbidity=0.0, low_light=0.0, blur_k=1):
    """The degradation utility used in the Aquatic Guardian project notebooks (copied verbatim in spirit)."""
    out = img.copy()
    if blur_k > 1:
        pad = blur_k // 2; P = np.pad(out, ((pad, pad), (pad, pad), (0, 0)), mode="edge"); acc = np.zeros_like(out)
        for dy in range(blur_k):
            for dx in range(blur_k): acc += P[dy:dy + out.shape[0], dx:dx + out.shape[1], :]
        out = acc / (blur_k * blur_k)
    if turbidity > 0:
        veil = np.array([0.20, 0.42, 0.45]); out = (1 - turbidity) * out + turbidity * veil
        out = (1 - 0.5 * turbidity) * out + 0.5 * turbidity * out.mean()
    if low_light > 0: out = (out * (1 - 0.85 * low_light)) ** (1 + 0.6 * low_light)
    return np.clip(out, 0, 1)

def show(images, titles=None, size=3.2, cmap=None):
    images = images if isinstance(images, (list, tuple)) else [images]
    fig, ax = plt.subplots(1, len(images), figsize=(size * len(images), size * 0.72))
    ax = np.atleast_1d(ax)
    for i, (a, im) in enumerate(zip(ax, images)):
        a.imshow(im, cmap=cmap if im.ndim == 2 else None, vmin=0 if im.ndim == 2 and cmap else None,
                 vmax=1 if im.ndim == 2 and cmap else None)
        a.set_title(titles[i] if titles else ""); a.axis("off")
    plt.tight_layout(); plt.show()

def draw_boxes(img, boxes, colour=(1.0, 0.85, 0.0)):
    out = img.copy()
    for (x1, y1, x2, y2) in boxes:
        x2 -= 1; y2 -= 1
        out[y1, x1:x2 + 1] = colour; out[y2, x1:x2 + 1] = colour
        out[y1:y2 + 1, x1] = colour; out[y1:y2 + 1, x2] = colour
    return out
print("Setup ready. Water types:", list(WATER))

In [ ]:
#@title ▶ Setup: a staged dataset of short camera clips (fish / non_fish / empty_frame)
CLASSES = ["fish", "non_fish", "empty_frame"]     # Mentee 1's Level-1 classes
HW = 48                                           # frame size in pixels
HABITATS = {   # background colour in air, water type, typical object distance (m)
    "reef":    dict(bg=(0.52, 0.47, 0.36), water="clear_ocean",    dist=(1.0, 4.0)),
    "kelp":    dict(bg=(0.30, 0.34, 0.22), water="coastal",        dist=(1.0, 3.5)),
    "harbour": dict(bg=(0.36, 0.33, 0.28), water="turbid_harbour", dist=(0.6, 2.2)),
}

def new_session(cls, habitat, r):
    h = HABITATS[habitat]
    return dict(cls=cls, habitat=habitat, water=h["water"], bg=np.array(h["bg"]) + r.normal(0, 0.03, 3),
                obj_dist=r.uniform(*h["dist"]), bg_dist=r.uniform(6, 12), photons=r.uniform(250, 900),
                x0=r.uniform(14, 34), y0=r.uniform(14, 34), vx=r.normal(0, 0.6), vy=r.normal(0, 0.3),
                L=r.uniform(16, 30), ang=r.uniform(0, np.pi), aspect=r.uniform(3.2, 4.5),
                col=np.array(FISH_COLOURS[r.integers(len(FISH_COLOURS))]) + r.normal(0, 0.05, 3),
                blob_col=r.uniform(0.25, 0.6, 3), blob_r=r.uniform(4, 8), blob_aspect=r.uniform(1.0, 1.5))

def render_frame(s, i, r):
    yy, xx = np.mgrid[0:HW, 0:HW] / HW
    J = s["bg"] + 0.06 * np.stack([yy, 0.8 * yy, 0.5 * xx], -1) + 0.03 * r.standard_normal((HW, HW, 1))
    dist = np.full((HW, HW), s["bg_dist"])
    cx = np.clip(s["x0"] + s["vx"] * i, 8, HW - 8); cy = np.clip(s["y0"] + s["vy"] * i, 8, HW - 8)
    if s["cls"] == "fish":
        m = _fish_mask(HW, HW, cx, cy, s["L"], s["ang"] + 0.02 * i, s["aspect"])[0]; J[m] = s["col"]; dist[m] = s["obj_dist"]
    elif s["cls"] == "non_fish":     # crab, rock, jellyfish: compact blobs
        m = _fish_mask(HW, HW, cx, cy, 2.25 * s["blob_r"], s["ang"], s["blob_aspect"])[0]
        m &= ((np.mgrid[0:HW, 0:HW][1] - cx) ** 2 + (np.mgrid[0:HW, 0:HW][0] - cy) ** 2) <= (1.2 * s["blob_r"]) ** 2
        J[m] = s["blob_col"]; dist[m] = s["obj_dist"]
    return underwater(np.clip(J, 0, 1), dist, s["water"], photons=s["photons"], seed=int(r.integers(1 << 30)))

def make_clips(n_sessions_per_class=40, frames_per_clip=12, habitats=("reef", "kelp", "harbour"), seed=0):
    """Each session is one short clip: one object (or none) drifting past a fixed camera.
    Returns images X (N, HW, HW, 3), labels y (N,), session ids (N,), habitat names (N,)."""
    r = np.random.default_rng(seed); X, y, sess, hab = [], [], [], []
    sid = 0
    for ci, cls in enumerate(CLASSES):
        for k in range(n_sessions_per_class):
            habitat = habitats[k % len(habitats)]
            s = new_session(cls, habitat, r)
            for i in range(frames_per_clip):
                X.append(render_frame(s, i, r)); y.append(ci); sess.append(sid); hab.append(habitat)
            sid += 1
    return np.array(X), np.array(y), np.array(sess), np.array(hab)
print("Clip generator ready. Classes:", CLASSES)

In [ ]:
import pandas as pd
X, y, sess, hab = make_clips(n_sessions_per_class=45, frames_per_clip=12, seed=0)
print("frames:", X.shape, "sessions:", len(np.unique(sess)))
print(pd.crosstab(pd.Series(hab, name="habitat"), pd.Series([CLASSES[i] for i in y], name="class")))
fig, ax = plt.subplots(1, 8, figsize=(12, 1.9))
first = np.where(sess == 3)[0][:8]
for a, k in zip(ax, first): a.imshow(X[k]); a.axis("off")
fig.suptitle("Eight consecutive frames from ONE clip: they are nearly identical", fontsize=9); plt.show()

## 1. The leakage trap ▶

Consecutive frames from one clip are near-duplicates. If some frames of a clip land in training and others in
testing, the test is partly a memory test. We show this with the laziest possible classifier,
**1-nearest-neighbour on raw pixels**: it labels a test frame with the label of the most similar training frame.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
flat = X[:, ::2, ::2].reshape(len(X), -1)              # downsample to 24x24 for speed
r = np.random.default_rng(1)

# (a) random split by FRAME
idx = r.permutation(len(X)); cut = int(0.7 * len(X))
tr_f, te_f = idx[:cut], idx[cut:]
# (b) split by SESSION (clip)
sessions = np.unique(sess); r.shuffle(sessions)
train_sessions = set(sessions[:int(0.7 * len(sessions))])
tr_s = np.array([i for i in range(len(X)) if sess[i] in train_sessions]); te_s = np.array([i for i in range(len(X)) if sess[i] not in train_sessions])

knn = KNeighborsClassifier(1)
acc_frame = knn.fit(flat[tr_f], y[tr_f]).score(flat[te_f], y[te_f])
acc_sess = knn.fit(flat[tr_s], y[tr_s]).score(flat[te_s], y[te_s])
print(f"1-NN accuracy, random split by frame:   {acc_frame:.3f}")
print(f"1-NN accuracy, split by session (clip): {acc_sess:.3f}")

The frame-split number is the one that gets reported by mistake. The session split is the honest estimate of
how the model does on a **new** camera visit. Rule for the whole project: **split by clip, session, cage or site —
never by frame.** From here on we use the session split.

## 2. Features: describing a frame with a few numbers ▶

The Mentee 1 baseline computes interpretable features: foreground **area**, **elongation**, mean **colour** of the
foreground, and **edge density**. Same idea here (our frames are dimmer, so the foreground threshold is lower).

In [ ]:
def features(img, fg_thresh=0.10):
    corners = np.concatenate([img[:6, :6].reshape(-1, 3), img[:6, -6:].reshape(-1, 3),
                              img[-6:, :6].reshape(-1, 3), img[-6:, -6:].reshape(-1, 3)])
    bgc = np.median(corners, 0)
    mask = np.sqrt(((img - bgc) ** 2).sum(-1)) > fg_thresh
    area = mask.mean()
    if mask.sum() >= 5:
        ys, xs = np.where(mask); ev = np.clip(np.linalg.eigvalsh(np.cov(np.stack([xs - xs.mean(), ys - ys.mean()]))), 1e-6, None)
        elong = np.sqrt(ev[1] / ev[0]); mr, mg, mb = img[mask].mean(0) - bgc
    else:
        elong = mr = mg = mb = 0.0
    g = img.mean(-1); edge = np.abs(np.diff(g, axis=1)).mean() + np.abs(np.diff(g, axis=0)).mean()
    return np.array([area, elong, mr, mg, mb, edge])
FEATS = ["area", "elongation", "dR", "dG", "dB", "edge"]

F = np.array([features(im) for im in X])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for ci, c in enumerate(CLASSES):
    ax[0].scatter(F[y == ci, 0], F[y == ci, 1], s=6, alpha=0.5, label=c)
ax[0].set_xlabel("area (fraction of frame)"); ax[0].set_ylabel("elongation"); ax[0].legend(fontsize=8)
ax[0].set_title("Two features already separate most frames")
pd.DataFrame(F, columns=FEATS).assign(cls=[CLASSES[i] for i in y]).groupby("cls").mean().round(3).plot.bar(ax=ax[1], legend=True)
ax[1].set_title("mean feature value by class"); ax[1].tick_params(axis="x", rotation=0); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 3. A softmax classifier from scratch ▶

For each class $k$ the model computes a score $z_k = \mathbf{w}_k \cdot \mathbf{x} + b_k$, then turns scores into
probabilities with softmax, $p_k = e^{z_k} / \sum_j e^{z_j}$. Training nudges $\mathbf{w}$ and $b$ downhill on the
**cross-entropy** loss, $-\log p_{\text{true class}}$, averaged over frames. Features are standardised first
(mean 0, standard deviation 1) using **training statistics only**.

In [ ]:
Ftr, Fte, ytr, yte = F[tr_s], F[te_s], y[tr_s], y[te_s]
mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-6
Ztr, Zte = (Ftr - mu) / sd, (Fte - mu) / sd

def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def train_softmax(Xs, ys, C=3, lr=0.5, epochs=400, l2=1e-3):
    n, d = Xs.shape; W = np.zeros((d, C)); b = np.zeros(C); Y = np.eye(C)[ys]; losses = []
    for _ in range(epochs):
        P = softmax(Xs @ W + b)
        losses.append(-np.log(P[np.arange(n), ys] + 1e-12).mean())
        W -= lr * (Xs.T @ (P - Y) / n + l2 * W); b -= lr * (P - Y).mean(0)
    return W, b, losses

W, b, losses = train_softmax(Ztr, ytr)
P = softmax(Zte @ W + b); pred = P.argmax(1)
plt.figure(figsize=(5, 2.6)); plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("Learning curve"); plt.show()
print(f"test accuracy (session split): {(pred == yte).mean():.3f}")

from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(max_iter=2000).fit(Ztr, ytr)
print(f"scikit-learn LogisticRegression on the same features: {sk.score(Zte, yte):.3f}")

## 4. The confusion matrix and per-class metrics ▶

Rows are the truth, columns the prediction. For one class:

- **Precision** = of the frames we *called* fish, how many were fish? (cost of false alarms)
- **Recall** = of the frames that *were* fish, how many did we find? (cost of misses)
- **F1** = their harmonic mean.

In [ ]:
def confusion(y_true, y_pred, n=3):
    M = np.zeros((n, n), int)
    for t, p in zip(y_true, y_pred): M[t, p] += 1
    return M

def class_metrics(M):
    rows = {}
    for i, c in enumerate(CLASSES):
        tp = M[i, i]; fp = M[:, i].sum() - tp; fn = M[i, :].sum() - tp
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        rows[c] = dict(precision=pr, recall=rc, f1=2 * pr * rc / (pr + rc) if pr + rc else 0.0, support=int(M[i].sum()))
    return pd.DataFrame(rows).T.round(3)

Mx = confusion(yte, pred)
fig, ax = plt.subplots(figsize=(4, 3.4)); ax.imshow(Mx, cmap="Blues")
for i in range(3):
    for j in range(3): ax.text(j, i, Mx[i, j], ha="center", va="center", color="white" if Mx[i, j] > Mx.max() / 2 else "black")
ax.set_xticks(range(3)); ax.set_xticklabels(CLASSES, rotation=25); ax.set_yticks(range(3)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); plt.tight_layout(); plt.show()
class_metrics(Mx)

✏️ **Interpret:** which error is more costly for each use?

- a **ship-strike alert** for a protected species (miss = animal hit; false alarm = vessel slows);
- an **aquaculture welfare flag** that sends a person to check a fish;
- a **population count** used in a permit decision.

There's no single right threshold. Write down which metric you'd prioritise for each, and why.

## 5. Why accuracy lies on imbalanced data ▶

Real camera traps are mostly empty. Below, a test set where 85% of frames are empty, and a "model" that always
says `empty_frame`.

In [ ]:
r = np.random.default_rng(5)
empty_idx = te_s[yte == 2]; other_idx = te_s[yte != 2]
keep_other = r.choice(other_idx, size=int(len(empty_idx) * 0.15 / 0.85), replace=False)
imb = np.concatenate([empty_idx, keep_other]); y_imb = y[imb]
lazy = np.full_like(y_imb, 2)
ours = softmax(((F[imb] - mu) / sd) @ W + b).argmax(1)
for name, p in [("always 'empty_frame'", lazy), ("our softmax model", ours)]:
    m = class_metrics(confusion(y_imb, p))
    print(f"{name:22s} accuracy {(p == y_imb).mean():.3f}   macro-F1 {m.f1.mean():.3f}   fish recall {m.loc['fish', 'recall']:.3f}")

Report **per-class** recall/precision and a **macro** average (each class counts equally), not just accuracy.

## 6. Abstention: saying "I don't know" ▶

The model's top probability is a (rough) confidence. Below a threshold we return `unknown` and send the frame for
human review instead of acting on weak evidence. **Coverage** is the fraction of frames we still decide on. This is
Mentee 1's Task 7, and it's how the project's safety rules avoid treating a weak detection as fact.

In [ ]:
maxp = P.max(1); rows = []
for thr in [0.34, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    keep = maxp >= thr
    rows.append(dict(threshold=thr, coverage=keep.mean(), accuracy_on_kept=(pred[keep] == yte[keep]).mean() if keep.any() else np.nan,
                     sent_to_review=int((~keep).sum())))
abst = pd.DataFrame(rows).round(3)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(abst.threshold, abst.coverage, "o-", label="coverage"); ax.plot(abst.threshold, abst.accuracy_on_kept, "s-", label="accuracy on kept")
ax.set_xlabel("confidence threshold"); ax.legend(); ax.set_title("Abstention trade-off"); plt.tight_layout(); plt.show()
abst

**Caution.** A classifier's probabilities are not automatically *calibrated* (a 0.9 doesn't necessarily mean
right 90% of the time). Check before you rely on the numbers:

In [ ]:
bins = np.linspace(0.33, 1, 8); which = np.digitize(maxp, bins) - 1
conf, acc = [], []
for k in range(len(bins) - 1):
    sel = which == k
    if sel.sum() >= 5: conf.append(maxp[sel].mean()); acc.append((pred[sel] == yte[sel]).mean())
plt.figure(figsize=(3.6, 3.4)); plt.plot([0.33, 1], [0.33, 1], "k--", lw=1, label="perfect calibration")
plt.plot(conf, acc, "o-", label="our model"); plt.xlabel("stated confidence"); plt.ylabel("actual accuracy"); plt.legend(fontsize=8)
plt.title("Reliability diagram"); plt.tight_layout(); plt.show()

### 🧩 Exercise — recall from a confusion matrix

Using `Mx` (rows = true, columns = predicted), compute the recall for `fish` (class 0) yourself, without calling
`class_metrics`. Store it in `fish_recall`.

In [ ]:
fish_recall = None   # YOUR CODE

def check_recall(v):
    exp = Mx[0, 0] / Mx[0].sum()
    if v is None: print("Not yet: replace None."); return
    if abs(v - exp) < 1e-9: print(f"✅ Correct: {exp:.3f}")
    elif abs(v - Mx[0, 0] / Mx[:, 0].sum()) < 1e-9: print("❌ That's precision (you divided by the column). Recall divides by the row: all TRUE fish.")
    else: print(f"❌ Expected {exp:.3f}.")
check_recall(fish_recall)

In [ ]:
#@title Solution (double-click to show)
fish_recall = Mx[0, 0] / Mx[0].sum()
check_recall(fish_recall)

## Check yourself

<details><summary>1. Your model scores 0.98 with a random frame split and 0.81 with a session split. Which do you report?</summary>The session split. The 0.98 is inflated by near-duplicate frames shared between train and test.</details>
<details><summary>2. Why standardise features with training-set statistics only?</summary>Using test statistics leaks information about the test set into training.</details>
<details><summary>3. A fish detector has precision 0.95 and recall 0.40. Describe it in plain words.</summary>When it says fish it's usually right, but it misses most fish.</details>
<details><summary>4. What does raising the abstention threshold do to coverage and to accuracy on kept frames?</summary>Coverage falls (more frames go to review); accuracy on the frames still decided usually rises.</details>

**Next:** Module 05, *Robustness and honest reporting* — the same model under turbidity, low light and a new
habitat, and how to write that up so nobody over-reads it.